# Solution 3 — MFA + Fine-tuning Wav2Vec2 (FIXE)

## Diagnostic des 5 problemes de ton notebook

| # | Probleme | Cause | Impact |
|---|----------|-------|--------|
| 1 | `TypeError: Memory.__init__() got unexpected argument 'bytes_limit'` | MFA 2.2.17 incompatible avec joblib 1.4+ sur Kaggle | MFA ne demarre jamais |
| 2 | Modeles `french_mfa.zip` et `.dict` = **0 bytes** | wget echoue silencieusement puis `mfa model download` crash aussi | MFA ne peut pas aligner |
| 3 | Fallback eSpeak echoue aussi | `phonemizer` pas installe sur Kaggle | Les "phonemes" = texte brut (`maisjesuisvivantemoi`) |
| 4 | 1000 samples → seulement 31 valides → 26 train / 2 test | Le texte brut ne match pas le vocab IPA (61 tokens) | Fine-tuning inutile |
| 5 | PER = 500-700% | Reference = texte vs Prediction = phonemes IPA | Comparaison impossible |

## Solution : utiliser `phonemizer` (eSpeak) comme fallback FIABLE

MFA ne fonctionne pas sur Kaggle (bug joblib). On installe `phonemizer` + `espeak-ng` correctement.

In [2]:
%%bash
# CELLULE 1 — Installer espeak-ng + phonemizer + dependances
apt-get install -y espeak-ng 2>&1 | tail -3
pip install -q phonemizer "transformers<=4.44" datasets accelerate evaluate jiwer soundfile librosa scikit-learn 2>&1 | tail -3

# Verifier
python3 -c "from phonemizer import phonemize; print('phonemizer OK:', phonemize('bonjour', language='fr-fr', backend='espeak'))"
python3 -c "import transformers; print('transformers', transformers.__version__)"
echo "OK tout installe" 


/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 35.9 MB/s eta 0:00:00
phonemizer OK: bɔ̃ʒuʁ 
transformers 4.44.0
OK tout installe


In [3]:
# =============================================================================
# CELLULE 2 — Charger modele + phonemiser le dataset avec eSpeak
# =============================================================================

import os, re, glob
import numpy as np, pandas as pd, soundfile as sf, librosa
from pathlib import Path
from phonemizer import phonemize
import torch
from transformers import AutoModelForCTC, Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

WORK = Path("/kaggle/working")
SR = 16_000
MAX = 500  # nombre d'audios

# ── Charger modele ───────────────────────────────────────
MID = "Cnam-LMSSC/wav2vec2-french-phonemizer"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

tok = Wav2Vec2CTCTokenizer.from_pretrained(MID)
fe  = Wav2Vec2FeatureExtractor.from_pretrained(MID)
proc = Wav2Vec2Processor(feature_extractor=fe, tokenizer=tok)
model = AutoModelForCTC.from_pretrained(MID).to(DEVICE).eval()
vocab = tok.get_vocab()
print(f"Modele charge — vocab={len(vocab)} phonemes")

# ── Trouver dataset ──────────────────────────────────────
FR = Path("/kaggle/input/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr")
if not FR.exists():
    candidates = glob.glob("/kaggle/input/**/fr/validated.tsv", recursive=True)
    if candidates: FR = Path(candidates[0]).parent
    else: raise FileNotFoundError("Dataset non trouve")

CLIPS = FR / "clips"
print(f"Dataset: {FR}")

df_v = pd.read_csv(FR / "validated.tsv", sep="\t")
df_o = pd.read_csv(FR / "other.tsv", sep="\t")
df = pd.concat([df_v, df_o], ignore_index=True)
df = df[["client_id","path","sentence"]].dropna()
df = df[df["sentence"].str.len() > 5]
df = df[df["path"].apply(lambda p: (CLIPS/p).exists())]
df = df.head(MAX).reset_index(drop=True)
print(f"{len(df)} audios")

# ── Phonemiser avec eSpeak (labels de REFERENCE) ────────
print("\nPhonemisation eSpeak...")
df["text_clean"] = df["sentence"].apply(
    lambda t: re.sub(r"\s+", " ", re.sub(r"[^\w\s'\-\u00e0-\u00ff\u0153\u00e6]", "", str(t).lower())).strip())

df["ref_phonemes"] = df["text_clean"].apply(
    lambda t: phonemize(t, language='fr-fr', backend='espeak', strip=True,
                        language_switch='remove-flags'))
print(f"OK {df['ref_phonemes'].notna().sum()} phonemises")

# ── Predire avec le modele sur chaque audio ──────────────
print("\nPredictions Wav2Vec2...")
predictions = []
audio_paths = []

for i, row in df.iterrows():
    try:
        mp3 = str(CLIPS / row["path"])
        audio, _ = librosa.load(mp3, sr=SR, mono=True)
        
        # Sauvegarder WAV
        wav_path = str(WORK / "audio_16k" / f"cv_{i:05d}.wav")
        os.makedirs(WORK / "audio_16k", exist_ok=True)
        sf.write(wav_path, audio, SR)
        
        # Prediction
        inputs = proc(audio, sampling_rate=SR, return_tensors="pt").input_values.to(DEVICE)
        with torch.no_grad():
            logits = model(inputs).logits
        pred_ids = torch.argmax(logits, dim=-1)
        pred_phonemes = proc.batch_decode(pred_ids)[0]
        
        predictions.append(pred_phonemes)
        audio_paths.append(wav_path)
        
        if (i+1) % 100 == 0: print(f"  {i+1}/{len(df)}")
    except Exception as e:
        predictions.append("")
        audio_paths.append("")
        if i < 3: print(f"  Erreur: {e}")

df["pred_phonemes"] = predictions
df["audio_path"] = audio_paths

# ── Filtrer les samples valides ──────────────────────────
df = df[df["pred_phonemes"] != ""].copy()
df = df[df["ref_phonemes"].str.len() > 0].copy()

df.to_csv(WORK / "full_results.csv", index=False)
print(f"\nOK {len(df)} samples avec reference + prediction")
print(df[["sentence", "ref_phonemes", "pred_phonemes"]].head(3).to_string())

Device: cuda


tokenizer_config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/561 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/309 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of the model checkpoint at Cnam-LMSSC/wav2vec2-french-phonemizer were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at Cnam-LMSSC/wav2vec2-french-phonemizer and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probab

Modele charge — vocab=61 phonemes
Dataset: /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr
500 audios

Phonemisation eSpeak...
OK 500 phonemises

Predictions Wav2Vec2...


2026-04-05 14:07:13.864954: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775398034.064573      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775398034.121855      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775398034.612776      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775398034.612817      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775398034.612820      55 computation_placer.cc:177] computation placer alr

  100/500
  200/500
  300/500
  400/500
  500/500

OK 498 samples avec reference + prediction
                                                                         sentence                                                        ref_phonemes                                                     pred_phonemes
0                                               Quelques anciens élèves notables.                                          kɛlkəz ɑ̃sjɛ̃z elɛv notabl                                      ty ɛlkəz ɑ̃sjɛ̃z elɛv notabl
1  La ferme est située sur la commune de Bucilly, dans le département de l'Aisne.  la fɛʁm ɛ sitye syʁ la kɔmyn də bjuːsɪli dɑ̃ lə depaʁtəmɑ̃ də lɛsn  la fɛʁm ɛ sitye syʁ la kɔmyn də bysili dɑ̃ lə depaʁtəmɑ̃ də lɛsn
2                     Lorsqu’il s’éveille, il est terrible ; il frappe, il venge.                            lɔʁskj sevɛj il ɛ tɛʁibl il fʁap il vɑ̃ʒ                         loʁskil sevɛj il ɛ tɛʁibl il fʁap il vɑ̃ʒ


In [4]:
# =============================================================================
# CELLULE 3 — AFFICHER CHAQUE AUDIO: texte + reference + prediction + PER
# =============================================================================

import pandas as pd
from pathlib import Path
import evaluate

WORK = Path("/kaggle/working")
df = pd.read_csv(WORK / "full_results.csv")
wer = evaluate.load("wer")

print(f"{'='*80}")
print(f"  {len(df)} AUDIOS — REFERENCE vs PREDICTION")
print(f"{'='*80}")
print()

per_list = []

for i, (_, r) in enumerate(df.iterrows()):
    ref = str(r["ref_phonemes"]).strip()
    pred = str(r["pred_phonemes"]).strip()
    
    # Calculer PER individuel
    try:
        per = wer.compute(predictions=[pred], references=[ref])
    except:
        per = -1
    per_list.append(per)
    
    print(f"  [{i+1:3d}] {r['path']}")
    print(f"        Texte     : {r['sentence']}")
    print(f"        Reference : {ref}")
    print(f"        Prediction: {pred}")
    print(f"        PER       : {per:.2%}" if per >= 0 else f"        PER       : N/A")
    print()

df["PER"] = per_list

# ── Resume ───────────────────────────────────────────────
valid_per = [p for p in per_list if p >= 0]
print(f"{'='*80}")
print(f"  RESUME")
print(f"{'='*80}")
print(f"  Nombre total    : {len(df)}")
print(f"  PER moyen       : {np.mean(valid_per):.2%}" if valid_per else "  PER moyen : N/A")
print(f"  PER median      : {np.median(valid_per):.2%}" if valid_per else "")
print(f"  PER min         : {min(valid_per):.2%}" if valid_per else "")
print(f"  PER max         : {max(valid_per):.2%}" if valid_per else "")
print(f"  Samples PER<50% : {sum(1 for p in valid_per if p < 0.5)}/{len(valid_per)}")
print(f"  Samples PER<25% : {sum(1 for p in valid_per if p < 0.25)}/{len(valid_per)}")
print(f"{'='*80}")

df.to_csv(WORK / "full_results_with_per.csv", index=False)
print(f"\nCSV sauvegarde: full_results_with_per.csv")

  498 AUDIOS — REFERENCE vs PREDICTION

  [  1] common_voice_fr_42766720.mp3
        Texte     : Quelques anciens élèves notables.
        Reference : kɛlkəz ɑ̃sjɛ̃z elɛv notabl
        Prediction: ty ɛlkəz ɑ̃sjɛ̃z elɛv notabl
        PER       : 50.00%

  [  2] common_voice_fr_43360250.mp3
        Texte     : La ferme est située sur la commune de Bucilly, dans le département de l'Aisne.
        Reference : la fɛʁm ɛ sitye syʁ la kɔmyn də bjuːsɪli dɑ̃ lə depaʁtəmɑ̃ də lɛsn
        Prediction: la fɛʁm ɛ sitye syʁ la kɔmyn də bysili dɑ̃ lə depaʁtəmɑ̃ də lɛsn
        PER       : 7.14%

  [  3] common_voice_fr_42727647.mp3
        Texte     : Lorsqu’il s’éveille, il est terrible ; il frappe, il venge.
        Reference : lɔʁskj sevɛj il ɛ tɛʁibl il fʁap il vɑ̃ʒ
        Prediction: loʁskil sevɛj il ɛ tɛʁibl il fʁap il vɑ̃ʒ
        PER       : 11.11%

  [  4] common_voice_fr_42927221.mp3
        Texte     : Lors des pleines lunes, le télésiège est en fonctionnement aussi de nuit.
        Ref

In [5]:
# TESTER 4 AUDIOS
import librosa, torch
from IPython.display import Audio, display

SR = 16_000

# VRAI CHEMIN (confirme par le debug)
CLIPS = "/kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips"

# Tes 4 audios — change les noms si tu veux
MES_AUDIOS = [
    "common_voice_fr_43386933.mp3",
    "common_voice_fr_43354798.mp3",
    "common_voice_fr_43353639.mp3",
    "common_voice_fr_42967672.mp3",
]

for i, fichier in enumerate(MES_AUDIOS):
    path = f"{CLIPS}/{fichier}"
    try:
        audio, _ = librosa.load(path, sr=SR, mono=True)
        
        inputs = proc(audio, sampling_rate=SR, return_tensors="pt").input_values.to(DEVICE)
        with torch.no_grad():
            logits = model(inputs).logits
        pred = proc.batch_decode(torch.argmax(logits, dim=-1))[0]
        conf = torch.softmax(logits, dim=-1).max(dim=-1).values[0].mean().item()
        
        print(f"[{i+1}] {fichier}  ({len(audio)/SR:.1f}s)")
        print(f"    Prediction: {pred}")
        print(f"    Confiance : {conf:.1%}")
        display(Audio(audio, rate=SR))
        print()
    except Exception as e:
        print(f"[{i+1}] ERREUR: {e}\n")

[1] common_voice_fr_43386933.mp3  (5.6s)
    Prediction: nu navɔ̃ pa də ʒøn daːm a bɔʁ ʁepɔ̃di lə pyʁsœʁ
    Confiance : 98.4%



[2] common_voice_fr_43354798.mp3  (8.3s)
    Prediction: il ɛ lə fis də pjɛʁ lyi buʁbje maʁʃɑ̃ e də sesil katʁin dyʁɑ
    Confiance : 98.9%



[3] common_voice_fr_43353639.mp3  (8.5s)
    Prediction: il ɛt eɡalmɑ̃ kɔny puʁ sɔ̃n ɛ̃plikasjɔ̃ o sɛ̃ dy paʁti sosjalist də ɡoʃ
    Confiance : 99.3%



[4] common_voice_fr_42967672.mp3  (7.8s)
    Prediction: katoʁyvjaznajo vot djəʁyspa kɔ̃tolovjazna
    Confiance : 94.8%


In [6]:
# CELLULE DEBUG — Trouver le vrai chemin des clips
import glob

# Chercher les MP3
mp3s = glob.glob("/kaggle/input/**/*.mp3", recursive=True)
print(f"{len(mp3s)} MP3 trouves")

if mp3s:
    print(f"\nChemin type: {mp3s[0]}")
    print(f"\n10 premiers:")
    for f in mp3s[:10]:
        print(f"  {f}")
else:
    # Chercher tout ce qui existe
    print("\nAucun MP3 ! Contenu de /kaggle/input/:")
    import os
    for root, dirs, files in os.walk("/kaggle/input"):
        level = root.replace("/kaggle/input", "").count(os.sep)
        if level < 4:
            print(f"  {'  '*level}{os.path.basename(root)}/")
            if level >= 2:
                for f in files[:5]:
                    print(f"  {'  '*(level+1)}{f}")
                if len(files) > 5:
                    print(f"  {'  '*(level+1)}... +{len(files)-5} fichiers")

8342 MP3 trouves

Chemin type: /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_43386933.mp3

10 premiers:
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_43386933.mp3
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_43354798.mp3
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_43353639.mp3
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42967672.mp3
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_43372726.mp3
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_43347800.mp3
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-p

In [7]:
# =============================================================================
# CELLULE 5 — PLUSIEURS AUDIOS DE TON CHOIX
# =============================================================================

import librosa, torch
from IPython.display import Audio, display
from phonemizer import phonemize

# ┌──────────────────────────────────────────────────────────────┐
# │  AJOUTE TES AUDIOS ET TEXTES ICI                           │
# └──────────────────────────────────────────────────────────────┘
AUDIOS = [
    {
        "path": "/kaggle/input/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42705592.mp3",
        "texte": "Quelques anciens eleves notables",
    },
    {
        "path": "/kaggle/input/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42708557.mp3",
        "texte": "",  # laisse vide si tu ne connais pas le texte
    },
    {
        "path": "/kaggle/input/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42708575.mp3",
        "texte": "",
    },
]

SR = 16_000
import evaluate
wer = evaluate.load("wer")

print(f"{'='*70}")
print(f"  {len(AUDIOS)} AUDIOS")
print(f"{'='*70}\n")

for i, item in enumerate(AUDIOS):
    try:
        audio, _ = librosa.load(item["path"], sr=SR, mono=True)
        inputs = proc(audio, sampling_rate=SR, return_tensors="pt").input_values.to(DEVICE)
        with torch.no_grad():
            logits = model(inputs).logits
        pred = proc.batch_decode(torch.argmax(logits, dim=-1))[0]
        conf = torch.softmax(logits, dim=-1).max(dim=-1).values[0].mean().item()

        fname = item["path"].split("/")[-1]
        print(f"  [{i+1}] {fname}  ({len(audio)/SR:.1f}s)")
        
        if item.get("texte"):
            ref = phonemize(item["texte"].lower(), language='fr-fr', backend='espeak', strip=True)
            per = wer.compute(predictions=[pred], references=[ref])
            print(f"      Texte     : {item['texte']}")
            print(f"      Ref eSpeak: {ref}")
            print(f"      Prediction: {pred}")
            print(f"      PER       : {per:.2%}")
        else:
            print(f"      Prediction: {pred}")
        
        print(f"      Confiance : {conf:.1%}")
        try: display(Audio(audio, rate=SR))
        except: pass
        print()
    except Exception as e:
        print(f"  [{i+1}] ERREUR: {e}\n")

print(f"{'='*70}")

  3 AUDIOS

  [1] ERREUR: [Errno 2] No such file or directory: '/kaggle/input/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42705592.mp3'

  [2] ERREUR: [Errno 2] No such file or directory: '/kaggle/input/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42708557.mp3'

  [3] ERREUR: [Errno 2] No such file or directory: '/kaggle/input/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42708575.mp3'



/tmp/ipykernel_55/1207254893.py:37: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(item["path"], sr=SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [8]:
# =============================================================================
# CELLULE 6 — Lister les audios disponibles
# Copie un chemin et colle-le dans AUDIO_PATH de la Cellule 4
# =============================================================================

import glob
mp3s = sorted(glob.glob("/kaggle/input/**/*.mp3", recursive=True))
print(f"{len(mp3s)} fichiers MP3 disponibles\n")
print("--- 30 premiers (copie le chemin pour Cellule 4/5) ---")
for f in mp3s[:30]:
    print(f"  {f}")
if len(mp3s) > 30:
    print(f"  ... et {len(mp3s)-30} de plus")

8342 fichiers MP3 disponibles

--- 30 premiers (copie le chemin pour Cellule 4/5) ---
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42697572.mp3
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42697573.mp3
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42697574.mp3
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42697575.mp3
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42697576.mp3
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42698217.mp3
  /kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips/common_voice_fr_42698218.mp3
  /kaggle/i

#test moi seul

In [ ]:
# =============================================================================
# CELLULE — TESTER MES PROPRES AUDIOS + DETECTION LIAISON
# =============================================================================
# 
# COMMENT UPLOADER TES AUDIOS SUR KAGGLE :
#   1. Dans ton notebook Kaggle, clique "Add Data" (en haut a droite)
#   2. Clique "Upload" → "New Dataset" 
#   3. Upload tes fichiers .wav ou .mp3
#   4. Le chemin sera: /kaggle/input/ton-nom-dataset/ton_fichier.wav
#   5. Copie le chemin dans MES_TESTS ci-dessous
#
# OU : utilise les audios du dataset Common Voice deja charge
# =============================================================================

import librosa, torch, re
import numpy as np
from IPython.display import Audio, display
from phonemizer import phonemize
import evaluate

SR = 16_000
CLIPS = "/kaggle/input/datasets/mohamedyacineouhadi/dataset-prblm/cv-corpus-22.0-delta-2025-06-20/fr/clips"

# ┌──────────────────────────────────────────────────────────────────────┐
# │                                                                      │
# │  AJOUTE TES TESTS ICI                                               │
# │                                                                      │
# │  Pour chaque test:                                                   │
# │    "audio" : chemin complet vers le fichier audio                   │
# │    "texte" : ce que la personne DEVAIT lire (obligatoire)           │
# │                                                                      │
# │  Tu peux utiliser:                                                   │
# │    - Un fichier du dataset: f"{CLIPS}/common_voice_fr_XXXXX.mp3"   │
# │    - Ton propre audio uploade: "/kaggle/input/mon-dataset/xxx.wav" │
# │                                                                      │
# └──────────────────────────────────────────────────────────────────────┘

MES_TESTS = [
    {
        "audio": f"{CLIPS}/common_voice_fr_43386933.mp3",
        "texte": "Nous n'avons pas de jeune dame à bord répondit le poursuiveur",
    },
    {
        "audio": f"{CLIPS}/common_voice_fr_43354798.mp3",
        "texte": "Il est le fils de Pierre Louis Bourbier marchand et de Cécile Catherine Durand",
    },
    {
        "audio": f"{CLIPS}/common_voice_fr_43353639.mp3",
        "texte": "Il est également connu pour son implication au sein du parti socialiste de gauche",
    },
    {
        "audio": f"{CLIPS}/common_voice_fr_42927221.mp3",
        "texte": "Lors des pleines lunes le télésiège est en fonctionnement aussi de nuit",
    },
    # ── AJOUTE TES PROPRES AUDIOS ICI ─────────────────────────
    # {
    #     "audio": "/kaggle/input/mon-dataset/mon_audio.wav",
    #     "texte": "les amis sont arrivés",
    # },
]

# ── Regles de liaison ────────────────────────────────────────
LIAISONS_OBLIGATOIRES = {
    # determinant + nom/adj commencant par voyelle
    r'\bles\s+[aeiouyéèêëàâîïôûùœ]': {"consonne": "z", "type": "det+nom", "obligatoire": True},
    r'\bdes\s+[aeiouyéèêëàâîïôûùœ]': {"consonne": "z", "type": "det+nom", "obligatoire": True},
    r'\bmes\s+[aeiouyéèêëàâîïôûùœ]': {"consonne": "z", "type": "det+poss", "obligatoire": True},
    r'\bses\s+[aeiouyéèêëàâîïôûùœ]': {"consonne": "z", "type": "det+poss", "obligatoire": True},
    r'\bces\s+[aeiouyéèêëàâîïôûùœ]': {"consonne": "z", "type": "det+dem", "obligatoire": True},
    r'\baux\s+[aeiouyéèêëàâîïôûùœ]': {"consonne": "z", "type": "prep+nom", "obligatoire": True},
    # pronom + verbe
    r'\bils\s+[aeiouyéèêëàâîïôûùœ]': {"consonne": "z", "type": "pron+verbe", "obligatoire": True},
    r'\belles\s+[aeiouyéèêëàâîïôûùœ]': {"consonne": "z", "type": "pron+verbe", "obligatoire": True},
    r'\bon\s+[aeiouyéèêëàâîïôûùœ]': {"consonne": "n", "type": "pron+verbe", "obligatoire": True},
    r'\ben\s+[aeiouyéèêëàâîïôûùœ]': {"consonne": "n", "type": "prep+nom", "obligatoire": True},
    # est + voyelle
    r'\best\s+[aeiouyéèêëàâîïôûùœ]': {"consonne": "t", "type": "verbe+attr", "obligatoire": True},
    # un + nom
    r'\bun\s+[aeiouyéèêëàâîïôûùœ]': {"consonne": "n", "type": "det+nom", "obligatoire": True},
}

PHONEME_LIAISON = {"z": "z", "t": "t", "n": "n"}

wer = evaluate.load("wer")

print("=" * 75)
print("  TEST MANUEL — PREDICTION + DETECTION LIAISON")
print("=" * 75)

for idx, test in enumerate(MES_TESTS):
    audio_path = test["audio"]
    texte = test["texte"]
    
    try:
        # Charger audio
        audio, _ = librosa.load(audio_path, sr=SR, mono=True)
        
        # Prediction Wav2Vec2
        inputs = proc(audio, sampling_rate=SR, return_tensors="pt").input_values.to(DEVICE)
        with torch.no_grad():
            logits = model(inputs).logits
        pred = proc.batch_decode(torch.argmax(logits, dim=-1))[0]
        conf = torch.softmax(logits, dim=-1).max(dim=-1).values[0].mean().item()
        
        # Reference eSpeak
        ref = phonemize(texte.lower(), language='fr-fr', backend='espeak', strip=True,
                       language_switch='remove-flags')
        
        # PER
        per = wer.compute(predictions=[pred], references=[ref])
        
        fname = audio_path.split("/")[-1]
        print(f"\n{'─'*75}")
        print(f"  [{idx+1}] {fname}  ({len(audio)/SR:.1f}s)")
        print(f"{'─'*75}")
        print(f"  Texte      : {texte}")
        print(f"  Reference  : {ref}")
        print(f"  Prediction : {pred}")
        print(f"  PER        : {per:.2%}")
        print(f"  Confiance  : {conf:.1%}")
        
        # Player audio
        display(Audio(audio, rate=SR))
        
        # ── DETECTION DE LIAISON ─────────────────────────────
        texte_lower = texte.lower()
        liaisons_trouvees = []
        
        for pattern, info in LIAISONS_OBLIGATOIRES.items():
            matches = list(re.finditer(pattern, texte_lower))
            for match in matches:
                context = match.group()
                cons = info["consonne"]
                
                # Verifier si la consonne de liaison est dans la prediction
                # Chercher le phoneme de liaison dans la prediction
                pred_has_liaison = PHONEME_LIAISON[cons] in pred
                ref_has_liaison = PHONEME_LIAISON[cons] in ref
                
                liaisons_trouvees.append({
                    "contexte": context,
                    "consonne": f"/{cons}/",
                    "type": info["type"],
                    "obligatoire": info["obligatoire"],
                    "dans_reference": ref_has_liaison,
                    "dans_prediction": pred_has_liaison,
                })
        
        if liaisons_trouvees:
            print(f"\n  LIAISONS DETECTEES:")
            for li in liaisons_trouvees:
                status_pred = "OUI" if li["dans_prediction"] else "NON"
                status_ref = "OUI" if li["dans_reference"] else "NON"
                oblig = "OBLIGATOIRE" if li["obligatoire"] else "optionnelle"
                
                if li["dans_prediction"]:
                    verdict = "LIAISON REALISEE"
                else:
                    verdict = "LIAISON ABSENTE" if li["obligatoire"] else "liaison absente (ok)"
                
                print(f"    {li['consonne']} {li['contexte']:<25s} [{li['type']:<12s}] "
                      f"{oblig:<12s} → {verdict}")
        else:
            print(f"\n  Pas de contexte de liaison obligatoire dans cette phrase.")
        
    except Exception as e:
        print(f"\n  [{idx+1}] ERREUR: {e}")

print(f"\n{'='*75}")
print(f"  TERMINE — {len(MES_TESTS)} audios testes")
print(f"{'='*75}")